# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library with full reference to the dataset components' `@id`s.

### Dataset Source
The dataset source is provided as a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object and print basic info
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Examine all available record sets in the dataset schema, along with their fields and columns, referencing each by `@id`.

> **Tip:** The `@id` is the unique identifier for each schema entity. All queries and extractions should reference `@id` values.

In [ ]:
# Show all record sets and their field/column @ids
print("Available Record Sets and Fields in Dataset:\n")
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id})")
    record_set_ids.append(rs.id)
    if rs.fields:
        for field in rs.fields:
            print(f"  Field: {field.name} (@id: {field.id})  type: {field.data_type}")
            if hasattr(field, 'column') and field.column:
                print(f"    Column: {field.column.name} (@id: {field.column.id})")
    print('-' * 60)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. Reference record set and field `@id`s as provided in the overview for more targeted analysis.

In [ ]:
# Extract data from each record set, reference by @id
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    # The records are dictionaries with field @id as keys
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns in {rs_id}:\n{df.columns.tolist()}")
    print(df.head(2))
    print('-' * 40)

# For this notebook we pick the first record set for further analysis
target_record_set_id = record_set_ids[0] if record_set_ids else None
if target_record_set_id:
    print(f"\nUsing record set: {target_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate how to filter, normalize, and group by field values—using field (column) `@id`s for all operations.


In [ ]:
import numpy as np

if target_record_set_id is not None:
    df = dataframes[target_record_set_id]
    # Find first numeric field (float or int)
    # Use field @ids only
    numeric_field_id = None
    group_field_id = None

    for field in dataset.metadata.get_record_set(target_record_set_id).fields:
        if field.data_type in ['Number', 'Float', 'Integer'] and field.id in df.columns:
            numeric_field_id = field.id
            break
    for field in dataset.metadata.get_record_set(target_record_set_id).fields:
        # Pick a string/categorical field as group field
        if field.data_type in ['Text'] and field.id in df.columns:
            group_field_id = field.id
            break

    if numeric_field_id:
        print(f"Will analyze numeric field: {numeric_field_id}")

        # Coerce field to numeric, handle non-numeric values
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # Simple filter: values > threshold
        threshold = df[numeric_field_id].median() if np.isfinite(df[numeric_field_id].median()) else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered to records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for analysis in this record set.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize the numeric field distribution and group averages if possible.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Barplot of group mean
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values().tail(10)
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_means.values, y=group_means.index, palette='Blues_r')
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion

- Successfully loaded and explored the FAIR^2 dataset (Croissant schema) with `mlcroissant`.
- All queries, analyses, and groupings referenced dataset `@id`s for full schema transparency.
- Numeric fields (such as protein abundances or molecular weights) can be filtered, normalized, and visualized cleanly by automated `@id` mapping.
- For further research, repeat this process for additional record sets or fields of interest, always referencing their `@id`.
